# MATH Benchmark Evaluation

Evaluate models on the [MATH benchmark](https://huggingface.co/datasets/lighteval/MATH) with checkpointing support.
Results are saved incrementally to `output_dir` so runs can be interrupted and resumed.

## Setup

In [7]:
import sys, os

# When running in Colab, clone the repo and add lib/ to the path
if "google.colab" in sys.modules:
    if not os.path.exists("cs224n-final-project"):
        !git clone https://github.com/anujjamwal/cs224n-final-project.git
    !cd cs224n-final-project && git checkout claude/math-eval-setup-cuuRV
    !cd cs224n-final-project && git pull
    sys.path.insert(0, "cs224n-final-project/lib")
    !pip install math-verify trl

    from google.colab import drive

    drive.mount('/content/drive')
    DATA_DIR="/content/drive/My Drive"

else:
    # Local: notebook lives inside lib/ already
    sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
    DATA_DIR="."

Already on 'claude/math-eval-setup-cuuRV'
Your branch is up to date with 'origin/claude/math-eval-setup-cuuRV'.
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 417 bytes | 417.00 KiB/s, done.
From https://github.com/anujjamwal/cs224n-final-project
   31adf76..7baa07e  claude/math-eval-setup-cuuRV -> origin/claude/math-eval-setup-cuuRV
Updating 31adf76..7baa07e
Fast-forward
 lib/eval/benchmarks.py | 4 ++--
 1 file changed, 2 insertions(+), 2 deletions(-)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from eval import load_math, run_eval, summarize_results
from eval.runner import GenerationMode
from custom_generate.generate import _sample
from trainer import prepare_base_model

## Configuration

Edit the cell below to configure the evaluation.

In [9]:
# ---- Edit these ----
MODEL_PATH = "anujjamwal/OpenMath-Nemotron-1.5B-hcot"
OUTPUT_DIR = os.path.join(DATA_DIR, "cs224n/eval/math-500-hcot")
BATCH_SIZE = 4
MAX_NEW_TOKENS = 4096
SPLIT = "test"
LEVELS = None        # e.g. [1, 2] to filter by difficulty
SUBJECTS = None      # e.g. ["Algebra", "Number Theory"]
DTYPE = torch.bfloat16
NEEDS_PREPARE = False  # Set True if model needs special tokens added

# Generation modes to evaluate
MODES = [
    GenerationMode(name="HCoT Prune", generate_fn=_sample, kwargs={"prune_aware": True}),
    # GenerationMode(name="Baseline", generate_fn=None, kwargs={"use_cache": True}),
]

## Load Model

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype=DTYPE, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

if NEEDS_PREPARE:
    model, tokenizer = prepare_base_model(model, tokenizer)

model.eval()
print(f"Model loaded: {MODEL_PATH} ({sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params)")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded: anujjamwal/OpenMath-Nemotron-1.5B-hcot (1543M params)


## Load Dataset

In [15]:
DATASET="qwedsacf/competition_math"
DATASET="HuggingFaceH4/MATH-500"
problems = load_math(split="test", subjects=SUBJECTS, levels=LEVELS, dataset_id=DATASET)
print(f"Loaded {len(problems)} problems")

# Preview
for p in problems[:3]:
    print(f"  [{p.id}] level={p.metadata['level']} subject={p.metadata['subject']}")
    print(f"    {p.problem[:80]}...")
    print(f"    expected: {p.expected_answer}")

KeyError: 'type'

## Run Evaluation

Results are checkpointed after each batch. If you interrupt and re-run this cell,
it will skip already-evaluated problems.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")

results = run_eval(
    model, tokenizer, problems, OUTPUT_DIR, MODES,
    batch_size=BATCH_SIZE,
    max_new_tokens=MAX_NEW_TOKENS,
)

Eval [HCoT Prune]:   0%|          | 0/3125 [00:00<?, ?it/s]

: 

## Summary

In [ ]:
import pandas as pd

summary = summarize_results(results)
print(f"Overall: {summary['correct']}/{summary['total']} ({summary['accuracy']:.1%})")
print()

# By mode
if summary.get("by_mode"):
    print("By mode:")
    for mode, stats in summary["by_mode"].items():
        print(f"  {mode}: {stats['correct']}/{stats['total']} ({stats['accuracy']:.1%})")
    print()

# By level
if summary.get("by_level"):
    print("By level:")
    for level in sorted(summary["by_level"].keys()):
        stats = summary["by_level"][level]
        print(f"  {level}: {stats['correct']}/{stats['total']} ({stats['accuracy']:.1%})")
    print()

# By subject
if summary.get("by_subject"):
    print("By subject:")
    for subj, stats in sorted(summary["by_subject"].items()):
        print(f"  {subj}: {stats['correct']}/{stats['total']} ({stats['accuracy']:.1%})")

## Visualize

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy by level
if summary.get("by_level"):
    levels = sorted(summary["by_level"].keys())
    accs = [summary["by_level"][l]["accuracy"] * 100 for l in levels]
    ax = axes[0]
    bars = ax.bar(levels, accs)
    ax.set_ylabel("Accuracy (%)")
    ax.set_title("Accuracy by Difficulty Level")
    ax.set_ylim(0, 100)
    for bar, v in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width()/2, v + 1, f"{v:.1f}%", ha="center", fontsize=9)

# Accuracy by subject
if summary.get("by_subject"):
    subjects = sorted(summary["by_subject"].keys())
    accs = [summary["by_subject"][s]["accuracy"] * 100 for s in subjects]
    ax = axes[1]
    bars = ax.barh(subjects, accs)
    ax.set_xlabel("Accuracy (%)")
    ax.set_title("Accuracy by Subject")
    ax.set_xlim(0, 100)
    for bar, v in zip(bars, accs):
        ax.text(v + 1, bar.get_y() + bar.get_height()/2, f"{v:.1f}%", va="center", fontsize=9)

plt.tight_layout()
plt.show()

## Inspect Individual Results

In [ ]:
# Load results as a DataFrame for exploration
from eval import load_results as load_eval_results

df = pd.DataFrame([{
    "id": r.problem_id,
    "mode": r.mode,
    "correct": r.correct,
    "expected": r.expected,
    "predicted": r.predicted,
    "generated_tokens": r.generated_tokens,
    "wall_time": r.wall_time,
    "level": r.metadata.get("level"),
    "subject": r.metadata.get("subject"),
} for r in results])

print(f"Total results: {len(df)}")
print(f"\nIncorrect predictions (sample):")
display(df[~df["correct"]].head(10))

In [ ]:
df.to_json(os.path.join(OUTPUT_DIR, 'cs224n/eval/math_hcot_output.json'), orient='records', lines=True)